# `quadbench`

This notebook reports a set of microbenchmarks of NumPy and CuPy elementwise
arithmetic, measured on one x86-64 CPU and two variants of the NVIDIA A100.
Timings were collected by `bench_v2.py` (CPU) and `bench_gpu.py` (GPU) and stored
as per-repeat microsecond samples in `.npz` archives; `clean_npz.py` converts each
archive into a tidy Parquet table, which this notebook reads together with the
metadata JSON written alongside it.

| run | subject | sizes | dtypes |
|---|---|---|---|
| `cpu_sweep` | the full dtype grid at three sizes | 3 | 13, incl. `quad-sleef`, x87 `longdouble` |
| `cpu_dense` | size resolved finely, four dtypes | 18 | 5 |
| `cpu_alloc` | destination cost, three modes | 6 | 4 |
| `cpu_f16` | why `float16` costs what it does | 3 | 3 |
| `cpu_dram50` | one size past every cache | 1 | 5 |
| `cpu_par0…3` | four concurrent pinned processes | 1 | 1 |
| `gpu_dense80` | size resolved finely on an A100-80GB | 16 | 3 |
| `gpu_sweep80` | the full op grid on an A100-80GB | 4 | 3 |
| `gpu_sweep` | the same grid on an A100-**40**GB | 4 | 3 |
| `gpu_host80` | host timer instead of CUDA events | 7 | 3 |

Every measured cell is defined by four factors:

- **`destination`** — `alloc` allocates a fresh result array, as an ordinary
  Python expression would. `out` writes into one preallocated buffer, reused on
  every call. `out_pool` (`cpu_alloc` only) cycles the destination through a pool
  of preallocated buffers whose total size exceeds L3, so the destination is
  cache-cold without being freshly allocated.
- **`cache_state`** — `hot` reuses a single operand pair, which therefore stays
  cache-resident; `cold` cycles through distinct pairs so every call streams its
  operands from memory.
- **`n`** — problem size. Arrays have shape `(n, 2, 2)`, so `n_elem = 4n`.
- **`implementation`** — the operand dtype. NumPy may promote before computing;
  the metadata records the true result dtype, which is used for byte accounting
  throughout in preference to the operand dtype.

Timings are per call — the harness divides out its inner repetition count — with
20–50 repeats per cell. All quantities reported below are medians over those
repeats.

The two A100s are not a mistake to apologise for. They have identical compute
(108 SMs, sm_80) and differ only in memory: 1 555 GB/s of HBM2 against
2 039 GB/s of HBM2e. Part 3 uses that as a controlled comparison.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import altair as alt
import polars as pl
from IPython.display import HTML, display

# Flip to "dark" for the dark palette; every colour below re-derives from it.
THEME = "light"

DATA_ROOT = Path("data")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

PALETTE = {
    "light": {
        "surface": "#fcfcfb", "plane": "#f9f9f7",
        "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
        "grid": "#e1e0d9", "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a"],
        "shade": "#9ec5f4",     # second shade of series 1, for dumbbells
        "context": "#c3c2b7",   # de-emphasis grey for "everything else"
        "ramp": ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                 "#256abf", "#184f95", "#0d366b"],
    },
    "dark": {
        "surface": "#1a1a19", "plane": "#0d0d0d",
        "ink": "#ffffff", "ink2": "#c3c2b7", "muted": "#898781",
        "grid": "#2c2c2a", "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70"],
        "shade": "#184f95",
        "context": "#52514e",
        "ramp": ["#0d366b", "#184f95", "#256abf", "#3987e5",
                 "#6da7ec", "#9ec5f4", "#cde2fb"],
    },
}
P = PALETTE[THEME]


@alt.theme.register("quadbench", enable=True)
def _quadbench_theme() -> dict:
    """Thin marks, hairline solid chrome, recessive axes, validated hues."""
    return {
        "config": {
            "background": P["surface"],
            "font": FONT,
            "view": {"stroke": None, "continuousWidth": 620, "continuousHeight": 300},
            "axis": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["muted"], "titleColor": P["ink2"],
                "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": 500,
                "titlePadding": 8,
                "domainColor": P["axis"], "domainWidth": 1,
                "tickColor": P["axis"], "tickSize": 4,
                "gridColor": P["grid"], "gridWidth": 1, "gridDash": [],
            },
            "legend": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["ink2"], "titleColor": P["ink2"],
                "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": 500,
                "symbolType": "circle", "symbolSize": 90, "orient": "top",
                "direction": "horizontal", "offset": 8, "titlePadding": 10,
            },
            "title": {
                "font": FONT, "color": P["ink"], "fontSize": 15, "fontWeight": 600,
                "subtitleFont": FONT, "subtitleColor": P["ink2"],
                "subtitleFontSize": 11.5, "subtitlePadding": 8,
                "anchor": "start", "offset": 14, "dy": -4,
            },
            "header": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["ink2"], "titleColor": P["ink2"],
                "labelFontSize": 11.5, "labelFontWeight": 600, "titleFontSize": 11,
            },
            "range": {
                "category": P["series"],
                "heatmap": P["ramp"],
                "ramp": P["ramp"],
            },
            "bar": {"cornerRadiusEnd": 4, "discreteBandSize": 15},
            "point": {"size": 95, "filled": True,
                      "stroke": P["surface"], "strokeWidth": 2, "opacity": 1},
            "line": {"strokeWidth": 2, "strokeCap": "round", "strokeJoin": "round"},
            "rule": {"strokeWidth": 1},
            "text": {"font": FONT, "fontSize": 11, "color": P["ink2"]},
        }
    }


def table_view(df: pl.DataFrame, label: str = "Table view") -> HTML:
    """The WCAG-clean twin of a chart: every plotted value, as text."""
    return HTML(
        f"<details style='font:12px {FONT};color:{P['ink2']};margin:2px 0 18px'>"
        f"<summary style='cursor:pointer;padding:4px 0'>{label} "
        f"({df.height:,} rows)</summary>{df._repr_html_()}</details>"
    )


def figure(chart, table: pl.DataFrame | None = None, label: str = "Table view"):
    display(chart)
    if table is not None:
        display(table_view(table, label))


# Altair's default renderer embeds the spec plus a jsdelivr script tag, so saved
# outputs need a network connection to draw. alt.renderers.enable("mimetype")
# emits the raw Vega-Lite JSON instead — smaller, offline, but it relies on the
# viewer (JupyterLab, VS Code) shipping its own Vega-Lite 6 renderer.
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
alt.data_transformers.enable("default", max_rows=20000)

DataTransformerRegistry.enable('default')

## Loading

`time_us` is the only directly measured quantity. Four derived quantities carry
the analysis:

- **ns/element** — `time_us * 1000 / n_elem`, comparable across problem sizes.
- **bytes/element** — `reads * operand_bytes + result_bytes`, where `reads` is 1
  for the unary operations (`sqrt`, `exp`, `cos`) and 2 otherwise, and
  `result_bytes` follows from the *recorded result dtype*. The distinction is
  material: `cos` on `int32` operands is computed in `float64`, so its traffic is
  4 bytes in and 8 bytes out.
- **GB/s** — bytes/element × `n_elem` / seconds. Effective bandwidth.
- **Gelem/s** — `n_elem` / seconds. Elements retired per second, independent of
  how wide each element is. Part 3 needs both of the last two, because on the GPU
  they turn out to be limited by different things.

In [2]:
# Storage width in bytes. Hardcoded rather than asked of the local NumPy: the runs
# were collected on x86 Linux, where longdouble is the 80-bit x87 type in a 16-byte
# slot — a machine reading this notebook may disagree.
ITEMSIZE = {
    "int8": 1, "int16": 2, "int32": 4, "int64": 8,
    "uint8": 1, "uint16": 2, "uint32": 4, "uint64": 8,
    "float16": 2, "float32": 4, "float64": 8, "float128": 16,
    "longdouble64": 16, "quad-sleef": 16,
    "QuadPrecDType(backend='sleef')": 16,
    "GPU fp16": 2, "GPU fp32": 4, "GPU fp64": 8,
}
UNARY = ["sqrt", "exp", "cos"]

DTYPE_ORDER = ["int8", "int16", "int32", "int64",
               "uint8", "uint16", "uint32", "uint64",
               "float16", "float32", "float64", "longdouble64", "quad-sleef"]
OP_ORDER = ["add", "mul", "div", "muladd", "muladd_accum", "muladd_fused",
            "sqrt", "exp", "cos",
            "matmul", "matmul_explicit", "matmul_explicit_fused"]
MIB = 2 ** 20


def load_run(run: str) -> tuple[pl.DataFrame, dict]:
    """Per-repeat samples, enriched with element counts, byte widths and rates."""
    run_dir = DATA_ROOT / run
    df = pl.read_parquet(run_dir / f"{run}.parquet")
    meta = json.loads((run_dir / f"{run}.json").read_text())

    if "n" not in df.columns:  # single-size runs drop the n field from their keys
        df = df.with_columns(pl.lit(meta["sizes"][0], dtype=pl.Int64).alias("n"))

    n_elem = {int(size): block["n_elem"] for size, block in meta["per_size"].items()}
    promotions = pl.DataFrame(
        [
            {"n": int(size), "operation": key.split("__")[0],
             "implementation": key.split("__")[1], "result_dtype": res}
            for size, block in meta["per_size"].items()
            for key, res in block.get("result_dtypes", {}).items()
        ],
        schema={"n": pl.Int64, "operation": pl.String,
                "implementation": pl.String, "result_dtype": pl.String},
    )

    # `x_via_f32` reads the same operands as `x`; strip the suffix before asking
    # whether the op is unary.
    base_op = pl.col("operation").str.replace("_via_f32", "")

    return (
        df.join(promotions, on=["n", "operation", "implementation"], how="left")
        .with_columns(
            pl.lit(run).alias("run"),
            pl.col("n").replace_strict(n_elem).alias("n_elem"),
            # the GPU harness records no promotions; nothing it runs promotes
            pl.col("result_dtype").fill_null(pl.col("implementation")),
        )
        .with_columns(
            pl.col("implementation").replace_strict(ITEMSIZE).alias("operand_bytes"),
            pl.col("result_dtype").replace_strict(ITEMSIZE).alias("result_bytes"),
            pl.when(base_op.is_in(UNARY)).then(1).otherwise(2).alias("reads"),
            (pl.col("result_dtype") != pl.col("implementation")).alias("promoted"),
        )
        .with_columns(
            (pl.col("reads") * pl.col("operand_bytes")
             + pl.col("result_bytes")).alias("b_elem"),
        )
        .with_columns(
            (pl.col("time_us") * 1e3 / pl.col("n_elem")).alias("ns_elem"),
            (pl.col("b_elem") * pl.col("n_elem")
             / (pl.col("time_us") * 1e-6) / 1e9).alias("gbs"),
            (pl.col("n_elem") / (pl.col("time_us") * 1e-6) / 1e9).alias("gelem_s"),
            (pl.col("n_elem") * pl.col("result_bytes") / MIB).alias("result_mib"),
        ),
        meta,
    )


KEYS = ["run", "n", "n_elem", "operation", "implementation",
        "destination", "cache_state"]


def cells(df: pl.DataFrame) -> pl.DataFrame:
    """One row per measured cell: median plus the interquartile spread."""
    return (
        df.group_by(KEYS)
        .agg(
            pl.col("time_us").median().alias("us"),
            pl.col("time_us").quantile(0.25).alias("us_lo"),
            pl.col("time_us").quantile(0.75).alias("us_hi"),
            pl.col("ns_elem").median().alias("ns_elem"),
            pl.col("gbs").median().alias("gbs"),
            pl.col("gelem_s").median().alias("gelem_s"),
            pl.col("b_elem").first(),
            pl.col("result_mib").first(),
            pl.col("result_dtype").first(),
            pl.col("promoted").first(),
            pl.col("time_us").is_not_null().sum().alias("samples"),
        )
        .sort(KEYS)
    )


RUNS = ["cpu_sweep", "cpu_dense", "cpu_alloc", "cpu_f16", "cpu_dram50",
        "gpu_sweep", "gpu_sweep80", "gpu_dense80", "gpu_host80"]
raw = {r: load_run(r) for r in RUNS}
meta = {r: m for r, (_, m) in raw.items()}
D = {r: cells(df) for r, (df, _) in raw.items()}
par = {i: cells(load_run(f"cpu_par{i}")[0]) for i in range(4)}

OUT_COLD = (pl.col("destination") == "out") & (pl.col("cache_state") == "cold")
OUT_HOT = (pl.col("destination") == "out") & (pl.col("cache_state") == "hot")

everything = pl.concat([D[r] for r in RUNS])
everything.head(3)

run,n,n_elem,operation,implementation,destination,cache_state,us,us_lo,us_hi,ns_elem,gbs,gelem_s,b_elem,result_mib,result_dtype,promoted,samples
str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,i64,f64,str,bool,u32
"""cpu_sweep""",500,2000,"""add""","""float16""","""alloc""","""cold""",12.653133,12.646402,12.669868,6.326566,0.948382,0.158064,6,0.003815,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float16""","""alloc""","""hot""",12.591369,12.580333,12.604403,6.295684,0.953034,0.158839,6,0.003815,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float16""","""out""","""cold""",12.596365,12.582998,12.617065,6.298182,0.952656,0.158776,6,0.003815,"""float16""",false,30


In [3]:
def stat_tile(label: str, value: str, note: str) -> str:
    return (
        f"<div style='flex:1 1 190px;min-width:170px;background:{P['surface']};"
        f"border:1px solid {P['grid']};border-radius:10px;padding:14px 16px'>"
        f"<div style='font:500 11.5px {FONT};color:{P['muted']};"
        f"letter-spacing:.02em'>{label}</div>"
        f"<div style='font:600 30px {FONT};color:{P['ink']};margin:6px 0 2px'>{value}</div>"
        f"<div style='font:400 11.5px {FONT};color:{P['ink2']}'>{note}</div></div>"
    )


measured = everything.filter(pl.col("us").is_not_null()).height
samples = sum(len(df) for df, _ in raw.values())
cpu_env = meta["cpu_dense"]["env"]
gpu_dev = meta["gpu_dense80"]["device"]

display(HTML(
    f"<div style='display:flex;gap:12px;flex-wrap:wrap;font:{FONT};"
    f"background:{P['plane']};padding:14px;border-radius:12px'>"
    + stat_tile("Cells measured", f"{measured:,}",
                f"of {everything.height:,} planned · {everything.height - measured} "
                f"skipped by the harness")
    + stat_tile("Timing samples", f"{samples:,}",
                "20–50 repeats per cell, medians throughout")
    + stat_tile("CPU", "x86-64 · 1 thread",
                f"AVX2 usable ({', '.join(cpu_env['cpu_dispatch_usable'])}), OpenBLAS")
    + stat_tile("GPU", gpu_dev["name"].replace("NVIDIA ", ""),
                f"{gpu_dev['sms']} SMs · {gpu_dev['peak_hbm_gbs']:,.0f} GB/s peak HBM")
    + "</div>"
))

---

## Part 1 — the CPU: what a dtype actually costs

`cpu_dense` resolves the size axis at 18 log-spaced points from 400 to 8 million
elements, which is what the rest of this section is built on. The earlier
`cpu_sweep` had three sizes, and three points cannot distinguish a curve from a
line.

The first figure is the per-element cost of `add` against problem size, with the
operands either cache-resident (`hot`) or streamed (`cold`).

In [4]:
dense = D["cpu_dense"]
FEATURED = ["float32", "float64", "int64"]

curve = (
    dense.filter((pl.col("destination") == "out")
                 & (pl.col("operation") == "add")
                 & pl.col("implementation").is_in(FEATURED))
    .select("implementation", "n_elem", "cache_state", "ns_elem", "us")
)

XN = alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
           axis=alt.Axis(format="~s", values=[1000, 10_000, 100_000, 1_000_000, 8_000_000]))
base = alt.Chart().mark_line(point=alt.OverlayMarkDef(size=45)).encode(
    x=XN,
    y=alt.Y("ns_elem:Q", title="ns / element", scale=alt.Scale(domainMin=0)),
    color=alt.Color("cache_state:N", sort=["hot", "cold"],
                    scale=alt.Scale(domain=["hot", "cold"], range=P["series"][:2]),
                    legend=alt.Legend(title=None)),
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("cache_state:N", title="operands"),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".3f")])

fig1 = base.properties(width=210, height=230).facet(
    column=alt.Column("implementation:N", title=None, sort=FEATURED),
    data=curve,
).resolve_scale(y="independent").properties(
    title=alt.Title("Per-element cost of add against problem size",
                    subtitle="cpu_dense · out= · 18 sizes · independent y-scales · "
                             "the minimum near 13 000 elements is L2, the rise past "
                             "400 000 is L3 giving out"),
)
figure(fig1, curve.sort("implementation", "n_elem"), "Table view — dense size curve")

alt.FacetChart(...)

implementation,n_elem,cache_state,ns_elem,us
str,i64,str,f64,f64
"""float32""",400,"""hot""",1.125794,0.450317
"""float32""",400,"""cold""",1.282994,0.513198
"""float32""",720,"""cold""",0.834411,0.600776
"""float32""",720,"""hot""",0.730305,0.52582
"""float32""",1280,"""cold""",0.540131,0.691367
"""float32""",1280,"""hot""",0.454123,0.581277
"""float32""",2240,"""hot""",0.316935,0.709934
"""float32""",2240,"""cold""",0.394676,0.884074
"""float32""",4000,"""hot""",0.235954,0.943814


Three features that the three-point sweep could not show.

The curve has **two knees, not one**. Cost falls to a minimum of 0.272 ns/element
at 12 800 elements (a 200 KB operand pair), rises to a shoulder across
40 000–128 000, rises again through 400 000–720 000, and then holds flat at
0.84–0.88 ns from 400 000 elements to 8 million. Those are the L2 and L3
boundaries, and the flat region beyond is DRAM.

The **`hot`/`cold` gap opens and then closes**. It is 1.34× at 12 800 elements,
peaks at 1.49× at 128 000, and is 0.99–1.02× at every size from 400 000 upward.
Once the working set is larger than cache, reusing one operand pair buys nothing,
because it was never going to stay resident either. An earlier version of this
notebook claimed the gap widens once arrays outgrow cache; it was reading a
three-point sweep whose largest size sat almost exactly at the peak of the gap.

**`exp` has no such structure at all.** It is 4.6 ns/element at every one of the
18 sizes, with `cold`/`hot` equal to 1.00 throughout — the arithmetic is so much
more expensive than the memory traffic that the memory system never becomes
visible.

In [5]:
f16 = (
    dense.filter(OUT_HOT & (pl.col("operation") == "add")
                 & pl.col("implementation").is_in(["float16", "float32", "float64"]))
    .select("implementation", "n_elem", "ns_elem", "us")
)
ends = f16.filter(pl.col("n_elem") == f16["n_elem"].max())

XF = alt.X("n_elem:Q", scale=alt.Scale(type="log", nice=False),
           title="elements (log)",
           axis=alt.Axis(format="~s", values=[400, 4000, 40_000, 400_000, 8_000_000]))
YF = alt.Y("ns_elem:Q", scale=alt.Scale(type="log"), title="ns / element (log)",
           axis=alt.Axis(values=[0.1, 0.3, 1, 3, 10], format="~g"))
CF = alt.Color("implementation:N", sort=["float16", "float32", "float64"],
               scale=alt.Scale(domain=["float16", "float32", "float64"],
                               range=[P["series"][1], P["series"][0], P["shade"]]),
               legend=None)

lines = alt.Chart(f16).mark_line(strokeWidth=2.2).encode(x=XF, y=YF, color=CF)
dots = lines.mark_point(size=55, filled=True).encode(
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".3f")])
labs = alt.Chart(ends).mark_text(align="left", dx=10, fontSize=11.5, fontWeight=500).encode(
    x=XF, y=YF, color=CF, text="implementation:N")

fig2 = (lines + dots + labs).properties(
    width=440, height=290,
    title=alt.Title("float16 is the one dtype with no cache behaviour",
                    subtitle="cpu_dense · add · hot, out= · float16 holds 6.0 ns/element "
                             "across a 20 000× range in size while float32 varies 8×"),
).configure_view(clip=False)
figure(fig2, f16.pivot("implementation", index="n_elem", values="ns_elem").sort("n_elem"),
       "Table view — float16 against float32 and float64")

alt.LayerChart(...)

n_elem,float16,float32,float64
i64,f64,f64,f64
400,7.124462,1.125794,1.188013
720,6.714335,0.730305,0.816545
1280,6.404842,0.454123,0.588261
2240,6.233343,0.316935,0.451853
4000,6.134953,0.235954,0.351973
7200,6.083089,0.186918,0.297975
12800,6.058553,0.151731,0.272105
22400,6.048948,0.137211,0.319562
40000,6.025237,0.149482,0.4429


`float16` costs 6.0 ns/element at 400 elements and 6.0 ns/element at 8 million,
varying by ±2% in between. It has no minimum, no shoulder, and no DRAM plateau,
because it never becomes limited by the memory system at any size. `float32`
underneath it traverses its whole cache curve, an 8× range.

One consequence is that "float16 is *N*× slower than float32" has no single
answer: the ratio runs from 6.3× at 400 elements to 44.1× at 22 400 and back to
14.0× at 8 million, entirely because of where `float32` happens to be on its own
curve. Any single number quoted for that ratio is a statement about the chosen
problem size.

### Where the float16 cost comes from

`float16` has no ALU on this CPU; NumPy computes it by widening to `float32`.
The obvious question is whether the cost is the conversion or the loop around it.
`cpu_f16` answers it directly with `add_via_f32`, which does the widening by hand
— `astype` up, operate, `astype` back, allocating three intermediates along the
way. If explicit widening were *faster* than the native ufunc, the ufunc's inner
loop would be the problem. It isn't.

In [6]:
mech = (
    D["cpu_f16"].filter((pl.col("destination") == "alloc")
                        & (pl.col("cache_state") == "hot")
                        & (pl.col("n_elem") == 256_000))
    .with_columns(pl.col("operation").str.replace("_via_f32", "").alias("op"),
                  pl.when(pl.col("operation").str.contains("via_f32"))
                    .then(pl.lit("widened by hand"))
                    .otherwise(pl.lit("native ufunc")).alias("path"))
    .select("op", "implementation", "path", "ns_elem")
)
PATHS = ["native ufunc", "widened by hand"]
span = (mech.pivot("path", index=["op", "implementation"], values="ns_elem")
        .rename({PATHS[0]: "native", PATHS[1]: "via"}))
# One dataset for both layers: a faceted layer cannot mix sources, because the
# inner data would override what the facet hands down.
mech = mech.join(span, on=["op", "implementation"])

# A dumbbell rather than bars: the values span 40x, so the axis has to be log,
# and a bar cannot baseline on a log scale.
SCALE_M = alt.Scale(type="log", domain=[0.1, 12], nice=False)
AXIS_M = alt.Axis(values=[0.1, 0.3, 1, 3, 10], format="~g")
XM = alt.X("ns_elem:Q", scale=SCALE_M, title="ns / element (log)", axis=AXIS_M)
YM = alt.Y("op:N", title=None, sort=["add", "mul", "exp"])
mrule = alt.Chart().mark_rule(color=P["context"], strokeWidth=2.5).encode(
    x=alt.X("native:Q", scale=SCALE_M, title="ns / element (log)", axis=AXIS_M),
    x2="via:Q", y=YM)
mdots = alt.Chart().mark_point(size=95, filled=True).encode(
    x=XM, y=YM,
    color=alt.Color("path:N", sort=PATHS,
                    scale=alt.Scale(domain=PATHS, range=P["series"][:2]),
                    legend=alt.Legend(title=None, orient="top")),
    tooltip=[alt.Tooltip("op:N"), alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("path:N", title="path"),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".3f")])

fig3 = alt.layer(mrule, mdots).properties(width=225, height=150).facet(
    column=alt.Column("implementation:N", title=None,
                      sort=["float16", "float32", "float64"]),
    data=mech,
).properties(
    title=alt.Title("Doing the widening by hand does not help float16",
                    subtitle="cpu_f16 · 256 000 elements · hot, allocating form · log x · "
                             "'widened by hand' = astype to float32, operate, astype back · "
                             "float32/float64 price the extra copies alone"),
)
figure(fig3, span.sort("implementation", "op"), "Table view — native against hand-widened")

alt.FacetChart(...)

op,implementation,native,via
str,str,f64,f64
"""add""","""float16""",5.995123,6.503375
"""exp""","""float16""",6.286156,5.742582
"""mul""","""float16""",6.013205,6.498326
"""add""","""float32""",0.20926,0.621674
"""exp""","""float32""",1.187344,1.445263
"""mul""","""float32""",0.202745,0.586508
"""add""","""float64""",0.376971,1.238281
"""exp""","""float64""",4.587549,5.143631
"""mul""","""float64""",0.392814,1.223603


For `float16`, the hand-widened path costs 6.50 ns/element against the native
ufunc's 6.00 — 8% *slower*, having done three extra array copies. The two paths
cost the same because they are the same work: the conversion is the expense, and
NumPy is already doing it internally.

The `float32` and `float64` panels calibrate that. There `promote_types` makes the
widening a no-op cast, so the via-path prices only the copies: 0.62 against 0.21
ns/element, i.e. about 0.14 ns per pass. If `float16` conversion were as cheap as
a copy, the widened `float16` path would land near 0.7 ns/element. It lands at
6.5. Conversion is costing roughly 2 ns per element per pass, some 15× a plain
copy, and that single number sets the price of every `float16` operation
regardless of what the operation is.

This CPU reports `F16C`, which converts eight halves per instruction, so the
hardware to do this cheaply is present and unused by both paths.

### The rest of the dtype grid

`cpu_sweep` carries all 13 dtypes, including the two that are not hardware
arithmetic at all: x87 `longdouble` and `quad-sleef`, a software 128-bit float.
Measuring everything against `float64` at one size puts the whole grid on one
axis.

In [7]:
FAMILY = (pl.when(pl.col("implementation") == "quad-sleef")
            .then(pl.lit("quad-sleef (software, 128-bit)"))
          .when(pl.col("implementation") == "longdouble64")
            .then(pl.lit("longdouble (x87, 80-bit)"))
          .when(pl.col("implementation") == "float16")
            .then(pl.lit("float16 (converted, no ALU)"))
          .otherwise(pl.lit("the 10 hardware dtypes")))
NAMED = ["quad-sleef (software, 128-bit)", "longdouble (x87, 80-bit)",
         "float16 (converted, no ALU)"]

f64 = (D["cpu_sweep"].filter(OUT_HOT & (pl.col("n") == 64_000)
                             & (pl.col("implementation") == "float64"))
       .select("operation", pl.col("ns_elem").alias("base")))
tax = (
    D["cpu_sweep"].filter(OUT_HOT & (pl.col("n") == 64_000))
    .join(f64, on="operation")
    .with_columns((pl.col("ns_elem") / pl.col("base")).alias("vs_f64"), FAMILY.alias("family"))
    .drop_nulls("vs_f64")
    .select("operation", "implementation", "family", "vs_f64", "ns_elem")
)
ctx, named = tax.filter(~pl.col("family").is_in(NAMED)), tax.filter(pl.col("family").is_in(NAMED))

XT = alt.X("vs_f64:Q", scale=alt.Scale(type="log", nice=False),
           title="cost relative to float64, same operation (log)",
           axis=alt.Axis(values=[0.25, 1, 4, 16, 64, 256],
                         labelExpr="format(datum.value, '.3~g') + '×'"))
YT = alt.Y("operation:N", sort=OP_ORDER, title=None)
unity = alt.Chart(pl.DataFrame({"x": [1.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(x=XT)
utext = alt.Chart(pl.DataFrame({"x": [1.0], "t": ["float64"]})).mark_text(
    align="center", dy=-6, fontSize=11, color=P["ink2"]).encode(
    x=XT, y=alt.value(0), text="t:N")
cdots2 = alt.Chart(ctx).mark_point(size=60, filled=True, color=P["context"], opacity=0.85).encode(
    x=XT, y=YT,
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("vs_f64:Q", title="× float64", format=".2f")])
ndots = alt.Chart(named).mark_point(size=140, filled=True).encode(
    x=XT, y=YT,
    color=alt.Color("family:N", sort=NAMED,
                    scale=alt.Scale(domain=NAMED, range=P["series"]),
                    legend=alt.Legend(title=None, orient="top", columns=1)),
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("vs_f64:Q", title="× float64", format=".1f"),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".2f")])
qlab = alt.Chart(named.filter(pl.col("implementation") == "quad-sleef")).mark_text(
    align="left", dx=11, fontSize=10.5, color=P["ink2"]).encode(
    x=XT, y=YT, text=alt.Text("vs_f64:Q", format=".0f"))

fig4 = (unity + utext + cdots2 + ndots + qlab).properties(
    width=520, height=300,
    title=alt.Title("What precision costs, per operation",
                    subtitle="cpu_sweep · n = 64 000 · hot, out= · grey = the ten hardware "
                             "dtypes · labels give quad's multiple of float64"),
).configure_view(clip=False)
figure(fig4, named.sort("operation", "implementation"),
       "Table view — the dtypes that are not hardware arithmetic")

alt.LayerChart(...)

operation,implementation,family,vs_f64,ns_elem
str,str,str,f64,f64
"""add""","""float16""","""float16 (converted, no ALU)""",9.218737,5.998391
"""add""","""longdouble64""","""longdouble (x87, 80-bit)""",5.765204,3.751268
"""add""","""quad-sleef""","""quad-sleef (software, 128-bit)""",44.256522,28.796561
"""cos""","""float16""","""float16 (converted, no ALU)""",0.649647,8.89775
"""cos""","""longdouble64""","""longdouble (x87, 80-bit)""",13.743143,188.230137
"""cos""","""quad-sleef""","""quad-sleef (software, 128-bit)""",9.992344,136.858086
"""div""","""float16""","""float16 (converted, no ALU)""",9.324223,6.005355
"""div""","""longdouble64""","""longdouble (x87, 80-bit)""",5.883748,3.789484
"""div""","""quad-sleef""","""quad-sleef (software, 128-bit)""",121.150679,78.028256


The grey cloud is the point of the chart: ten hardware dtypes, every operation,
all within a small factor of `float64`. That is what "the FPU does this" looks
like, and the differences inside the cloud are mostly lane count.

Three series sit outside it, for three different reasons. `quad-sleef` is 27–121×
`float64` — software floating point in 128 bits, an order of magnitude and a half
of arithmetic. `longdouble` is 3–12×: real hardware, but x87 scalar hardware that
no SIMD unit will touch. And `float16` is out there too, 4–28× depending on the
operation, not because it lacks precision but because it is paying the conversion
cost from the previous figure.

Quad's multiple is *smallest* on `div` and the transcendentals — 27× on `exp`
against 44× on `add` — not because quad is comparatively fast there, but because
`float64` division and `exp` are themselves slow enough to narrow the ratio. Both
ends of a ratio move.

---

## Part 2 — the destination: allocation, and a cliff at 32 MiB

Every cell was measured in an allocating form and an `out=` form. Subtracting them
was meant to price the allocator, and at large sizes the difference goes negative:
allocating is *faster* than writing into a buffer you already own. That result
confounds two mechanisms pulling in opposite directions, so `cpu_alloc` adds a
third destination mode, `out_pool`, which cycles the destination through a 2 GB
pool of preallocated buffers. The destination is then cache-cold without being
freshly allocated, and the difference splits:

    out_pool − out       cost of a destination that has fallen out of cache
    alloc    − out_pool  cost of allocating and faulting a new one

In [8]:
alloc = D["cpu_alloc"]
three = (
    alloc.filter((pl.col("cache_state") == "cold") & (pl.col("operation") == "add"))
    .pivot("destination", index=["implementation", "n_elem"], values="us")
    .with_columns(((pl.col("out_pool") - pl.col("out")) / pl.col("out") * 100)
                  .alias("cold destination  (out_pool − out)"),
                  ((pl.col("alloc") - pl.col("out_pool")) / pl.col("out") * 100)
                  .alias("allocation  (alloc − out_pool)"))
    .unpivot(["cold destination  (out_pool − out)", "allocation  (alloc − out_pool)"],
             index=["implementation", "n_elem"],
             variable_name="component", value_name="pct")
    .sort("implementation", "n_elem")
)
COMP = ["cold destination  (out_pool − out)", "allocation  (alloc − out_pool)"]

zero = alt.Chart(pl.DataFrame({"y": [0.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(y="y:Q")
clines = alt.Chart(three).mark_line(point=alt.OverlayMarkDef(size=50)).encode(
    x=alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
            axis=alt.Axis(format="~s", values=[16_000, 100_000, 1_000_000, 8_000_000])),
    y=alt.Y("pct:Q", title="% of the out= time"),
    color=alt.Color("component:N", sort=COMP,
                    scale=alt.Scale(domain=COMP, range=P["series"][:2]),
                    legend=alt.Legend(title=None, orient="top", columns=1)),
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("component:N", title="component"),
             alt.Tooltip("pct:Q", title="% of out=", format=".1f")])

fig4 = (zero + clines).properties(width=205, height=220).facet(
    column=alt.Column("implementation:N", title=None,
                      sort=["float32", "float64", "int32", "int64"]),
    data=three,
).properties(
    title=alt.Title("The two halves of the allocation difference",
                    subtitle="cpu_alloc · add · cold operands · a cold destination costs "
                             "40–60% while the result still fits in cache, and nothing "
                             "once it does not"),
)
figure(fig4, three, "Table view — destination cost decomposition")

alt.FacetChart(...)

implementation,n_elem,component,pct
str,i64,str,f64
"""float32""",16000,"""allocation (alloc − out_pool)""",-45.444693
"""float32""",16000,"""cold destination (out_pool − …",46.702809
"""float32""",64000,"""allocation (alloc − out_pool)""",-60.447748
"""float32""",64000,"""cold destination (out_pool − …",61.096277
"""float32""",256000,"""cold destination (out_pool − …",22.719089
"""float32""",256000,"""allocation (alloc − out_pool)""",-33.802637
"""float32""",1024000,"""cold destination (out_pool − …",2.278915
"""float32""",1024000,"""allocation (alloc − out_pool)""",-5.788816
"""float32""",4000000,"""allocation (alloc − out_pool)""",-2.507312


The blue series is read-for-ownership, and it is large where it is present: a
destination that has fallen out of cache costs 47% on `float32` at 16 000
elements and 61% at 64 000. By 1 million elements it is 2%, and past that it is
zero — once the single `out` buffer is itself too large to stay resident, cycling
through a pool changes nothing about it.

The orange series is what allocation itself costs, and below 8 million elements it
is *negative*: allocating a fresh buffer is cheaper than reusing a pooled one.
That is the residue the decomposition does not explain — it is not the
destination's cache state, because `out_pool ≈ out` at exactly the sizes where it
appears.

At 8 million elements the orange series jumps to +34%, and that jump has a precise
cause.

In [9]:
def penalty(run: str, ops=("add", "mul")) -> pl.DataFrame:
    return (
        D[run].filter((pl.col("cache_state") == "cold") & pl.col("operation").is_in(ops))
        .pivot("destination", index=["operation", "implementation", "result_mib"],
               values="us")
        .drop_nulls(["alloc", "out"])
        .with_columns(((pl.col("alloc") - pl.col("out")) / pl.col("out") * 100).alias("pct"),
                      pl.lit(run).alias("run"))
        .select("run", "operation", "implementation", "result_mib", "pct")
    )


cliff = pl.concat([penalty("cpu_dense"), penalty("cpu_alloc"),
                   penalty("cpu_dram50", ops=("add", "mul", "div", "sqrt", "exp"))])
cliff = cliff.with_columns(
    pl.when(pl.col("result_mib") > 32).then(pl.lit("above 32 MiB"))
      .otherwise(pl.lit("at or below 32 MiB")).alias("side"))
SIDE = ["at or below 32 MiB", "above 32 MiB"]

thresh = alt.Chart(pl.DataFrame({"x": [32.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1, strokeDash=[4, 3]).encode(x="x:Q")
thresh_t = alt.Chart(pl.DataFrame({"x": [32.0], "t": ["glibc mmap threshold cap, 32 MiB"]})).mark_text(
    align="right", dx=-7, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(12), text="t:N")
zero2 = alt.Chart(pl.DataFrame({"y": [0.0]})).mark_rule(color=P["grid"]).encode(y="y:Q")
cdots = alt.Chart(cliff).mark_point(size=95, filled=True, opacity=0.85).encode(
    x=alt.X("result_mib:Q", scale=alt.Scale(type="log"), title="result buffer, MiB (log)",
            axis=alt.Axis(values=[0.05, 0.5, 5, 32, 61], format="~g")),
    y=alt.Y("pct:Q", title="allocating form, % slower than out="),
    color=alt.Color("side:N", sort=SIDE,
                    scale=alt.Scale(domain=SIDE, range=[P["context"], P["series"][1]]),
                    legend=alt.Legend(title=None, orient="top")),
    tooltip=[alt.Tooltip("run:N"), alt.Tooltip("operation:N", title="op"),
             alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("result_mib:Q", title="result MiB", format=".2f"),
             alt.Tooltip("pct:Q", title="% vs out=", format=".1f")])

fig5 = (zero2 + cdots + thresh + thresh_t).properties(
    width=500, height=300,
    title=alt.Title("The allocator only shows up past 32 MiB",
                    subtitle="cold operands · three runs pooled · below the threshold malloc "
                             "recycles heap memory, above it every call mmaps and munmaps"),
)
figure(fig5, cliff.sort("result_mib"), "Table view — allocation penalty by buffer size")

alt.LayerChart(...)

run,operation,implementation,result_mib,pct,side
str,str,str,f64,f64,str
"""cpu_dense""","""add""","""float16""",0.000763,-0.497808,"""at or below 32 MiB"""
"""cpu_dense""","""mul""","""float16""",0.000763,-0.490409,"""at or below 32 MiB"""
"""cpu_dense""","""add""","""float16""",0.001373,2.187781,"""at or below 32 MiB"""
"""cpu_dense""","""mul""","""float16""",0.001373,3.307509,"""at or below 32 MiB"""
"""cpu_dense""","""add""","""float32""",0.001526,10.558793,"""at or below 32 MiB"""
"""cpu_dense""","""mul""","""float32""",0.001526,9.612344,"""at or below 32 MiB"""
"""cpu_dense""","""add""","""float16""",0.002441,1.545053,"""at or below 32 MiB"""
"""cpu_dense""","""mul""","""float16""",0.002441,2.622518,"""at or below 32 MiB"""
"""cpu_dense""","""add""","""float32""",0.002747,10.108089,"""at or below 32 MiB"""


The step is the whole finding. The 223 cells whose result is at or below
30.52 MiB have a median penalty of 0.0% and a 90th percentile of 5.2%; the 21
cells at 61.04 MiB have a median of 34.4% and a minimum of 11.5%. The ranges do
graze each other — eight sub-threshold cells exceed +10% — but every one of those
eight has a result buffer of a few KiB, where the fixed cost of making a call
dominates and no memory effect is in play.

The boundary is glibc's `DEFAULT_MMAP_THRESHOLD_MAX` of 32 MiB. Below it `malloc`
satisfies the request from the heap, recycling the block the previous call just
freed, so there is no syscall and no page fault. Above it every call `mmap`s a
fresh region and `munmap`s it again, and the fresh region costs one first-touch
fault per page — 2 313 µs for a 64 MiB buffer, or 141 ns per 4 KiB page.

This supersedes an earlier reading of the same effect. `cpu_dram` showed 4-byte
results paying ~12% and 8-byte results ~46%, which was attributed to an 8-byte
result having twice as many pages to fault. The width was a proxy: at that one
problem size, 4-byte results are 30.52 MiB and 8-byte results are 61.04 MiB, so
the dtype was standing in for which side of the threshold the buffer landed on.
Resolving the size axis separates them, and it is the threshold that predicts the
cost.

The practical form of this is narrow: it is not that allocation is cheap, but
that the allocator's behaviour changes discontinuously at a tunable boundary, and
a benchmark that samples one problem size cannot see it.

### The bandwidth plateau, calibrated

`cpu_dram50` measures 8 million elements past every cache. The cheap elementwise
operations converge on one number regardless of dtype, which raises the question
of whether that number is a property of the memory system or of one core.

In [10]:
band = (
    D["cpu_dram50"].filter(OUT_COLD).drop_nulls("gbs")
    .with_columns(pl.format("{} · {}", pl.col("operation"), pl.col("implementation")).alias("cell"))
    .sort("gbs", descending=True)
)
plateau = band["gbs"].max()
band = band.with_columns(
    pl.when(pl.col("gbs") > plateau * 0.9).then(pl.lit("at the plateau"))
      .otherwise(pl.lit("short of it — compute-bound")).alias("state"))
STATE = ["at the plateau", "short of it — compute-bound"]
aggregate = sum(par[i].filter(OUT_COLD & (pl.col("operation") == "add"))["gbs"][0]
                for i in range(4))

bw = alt.Chart(band).mark_bar(size=8).encode(
    y=alt.Y("cell:N", sort="-x", title=None, axis=alt.Axis(labelLimit=180)),
    x=alt.X("gbs:Q", title="effective bandwidth, GB/s"),
    color=alt.Color("state:N", sort=STATE,
                    scale=alt.Scale(domain=STATE, range=[P["series"][0], P["context"]]),
                    legend=alt.Legend(title=None, orient="top", columns=1)),
    tooltip=[alt.Tooltip("cell:N", title="cell"),
             alt.Tooltip("result_dtype:N", title="computes in"),
             alt.Tooltip("gbs:Q", title="GB/s", format=".1f"),
             alt.Tooltip("us:Q", title="us/call", format=".0f")])
marks = pl.DataFrame({
    "x": [plateau, aggregate],
    "t": [f"{plateau:.1f} GB/s — one core", f"{aggregate:.1f} GB/s — four cores together"]})
rules = alt.Chart(marks).mark_rule(color=P["ink2"], strokeWidth=1).encode(x="x:Q")
rtext = alt.Chart(marks).mark_text(
    align="right", dx=-5, dy=-4, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(6), text="t:N")

fig6 = (bw + rules + rtext).properties(
    width=430, height=430,
    title=alt.Title("One core reaches 70% of what four cores reach",
                    subtitle="cpu_dram50 · 8M elements · cold, out= · the four-core figure "
                             "is cpu_par0…3, four pinned concurrent processes"),
)
figure(fig6, band.select("operation", "implementation", "result_dtype", "gbs", "us"),
       "Table view — effective bandwidth")

alt.LayerChart(...)

operation,implementation,result_dtype,gbs,us
str,str,str,f64,f64
"""mul""","""int64""","""int64""",27.942843,6871.191028
"""add""","""float64""","""float64""",27.776315,6912.385987
"""add""","""int64""","""int64""",27.660742,6941.246509
"""add""","""float32""","""float32""",27.441108,3498.419013
"""mul""","""float32""","""float32""",27.362795,3508.414025
"""mul""","""float64""","""float64""",27.320031,7027.822954
"""div""","""float64""","""float64""",27.211619,7055.817521
"""mul""","""int32""","""int32""",27.145048,3536.563017
"""add""","""int32""","""int32""",27.084691,3544.438048


Ten cells across four dtypes land between 27.0 and 27.9 GB/s. Running four
single-threaded copies pinned to separate cores gives 10.45, 9.73, 9.46 and
9.87 GB/s — an aggregate of 39.5 GB/s, only 1.42× the single-core figure. One core
therefore reaches 70% of what four reach together, and each individual process
slows by 2.8× when the others are running.

That makes the plateau a property of the memory system rather than of the core,
which is what the original claim needed and did not have. It also bounds it: the
socket is not going to give much more than 40 GB/s for this access pattern, so
the ops sitting at 27.7 are close to the practical ceiling and the ones below it
— `exp`, `div`, the matmuls — are the only ones where arithmetic is the variable.

---

## Part 3 — the GPU: two ceilings

`gpu_dense80` resolves the size axis at 16 points from 1 000 to 250 million
elements on an A100-80GB. The first thing it shows is that most of the range is
not measuring the device at all.

In [11]:
PEAK80 = meta["gpu_dense80"]["peak_hbm_gbs"]
sat = (
    D["gpu_dense80"].filter(OUT_COLD & (pl.col("operation") == "add"))
    .with_columns((pl.col("gbs") / PEAK80 * 100).alias("pct"))
    .select("implementation", "n_elem", "us", "gbs", "pct")
)
gpu_order = ["GPU fp64", "GPU fp32", "GPU fp16"]
floor_band = alt.Chart(pl.DataFrame({"x1": [900.0], "x2": [400_000.0]})).mark_rect(
    color=P["grid"], opacity=0.55).encode(x="x1:Q", x2="x2:Q")
XS = alt.X("n_elem:Q", scale=alt.Scale(type="log", nice=False), title="elements (log)",
           axis=alt.Axis(format="~s", values=[1000, 10_000, 400_000, 10_000_000, 250_000_000]))
sline = alt.Chart(sat).mark_line(point=alt.OverlayMarkDef(size=45)).encode(
    x=XS,
    y=alt.Y("pct:Q", title=f"% of the {PEAK80:,.0f} GB/s HBM peak",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("implementation:N", sort=gpu_order,
                    scale=alt.Scale(domain=gpu_order, range=P["series"]),
                    legend=alt.Legend(title=None, orient="top")),
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("us:Q", title="us/call", format=".2f"),
             alt.Tooltip("pct:Q", title="% of peak", format=".1f")])
band_t = alt.Chart(pl.DataFrame({"x": [20_000.0], "t": ["per-call cost is flat here: 8.4–9.3 µs"]})).mark_text(
    align="center", fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(20), text="t:N")

fig7 = (floor_band + sline + band_t).properties(
    width=490, height=300,
    title=alt.Title("An A100 does nothing measurable below 400 000 elements",
                    subtitle="gpu_dense80 · add · cold, out= · fp16 plateaus at half the "
                             "bandwidth utilisation of fp64, and stays there"),
)
figure(fig7, sat.sort("implementation", "n_elem"), "Table view — HBM saturation")

alt.LayerChart(...)

implementation,n_elem,us,gbs,pct
str,i64,f64,f64,f64
"""GPU fp16""",1000,9.300363,0.645136,0.031639
"""GPU fp16""",2000,8.732,1.374367,0.067403
"""GPU fp16""",4000,9.289333,2.583628,0.126708
"""GPU fp16""",10000,9.208,6.516087,0.319566
"""GPU fp16""",20000,8.769454,13.683862,0.671093
"""GPU fp16""",40000,8.697333,27.595137,1.35334
"""GPU fp16""",100000,9.089333,66.011853,3.237399
"""GPU fp16""",200000,8.597091,139.582326,6.845492
"""GPU fp16""",400000,8.8216,272.0603,13.342568


From 1 000 to 400 000 elements — a 400× range — the per-call cost is 8.4 to
9.3 µs and does not move, identically for all three precisions. Running the same
sizes with a host timer instead of CUDA events (`gpu_host80`) agrees to within
±1 µs, so this is not synchronisation the event timer was hiding.

What does move the floor is the number of array arguments: `exp` (one read, one
write) sits about 0.8 µs below `add` (two reads, one write) at every size and
every precision. A fixed cost that scales with argument count and not with data
volume or dtype is per-call dispatch in CuPy's Python layer, not kernel launch
latency and not memory traffic.

Above 400 000 elements the three precisions separate and then flatten at
different heights: fp64 at 87% of peak, fp32 at 80%, and fp16 at 50%, still 50%
at 250 million elements. fp16 does not merely need more elements to saturate the
bus — it never saturates it.

### Why fp16 stops at half

Plotting achieved bandwidth against how many bytes each element moves explains
that, and explains the rest of the run with it.

In [12]:
def ceilings(run: str) -> pl.DataFrame:
    m = meta[run]
    return (
        D[run].filter(OUT_HOT & (pl.col("n") == 25_000_000)
                      & ~pl.col("operation").str.starts_with("matmul")
                      & (pl.col("operation") != "muladd"))
        .select("operation", "implementation", "b_elem", "us", "gbs", "gelem_s")
        .with_columns(pl.lit(f"{m['device']['name'].replace('NVIDIA A100-SXM4-', '')}"
                             f"  ({m['peak_hbm_gbs']:,.0f} GB/s)").alias("card"))
    )


two = pl.concat([ceilings("gpu_sweep"), ceilings("gpu_sweep80")])
CARDS = sorted(two["card"].unique())
limits = (two.group_by("card")
            .agg(pl.col("gelem_s").filter(pl.col("gelem_s") > 150).mean().alias("elem"),
                 pl.col("gbs").max().alias("bw")))
model = pl.concat([
    pl.DataFrame({"b_elem": [b], "card": [r["card"]],
                  "gbs": [min(r["bw"], r["elem"] * b)]})
    for r in limits.iter_rows(named=True) for b in range(2, 27)
])

CSCALE = alt.Scale(domain=CARDS, range=P["series"][:2])
XB = alt.X("b_elem:Q", title="bytes moved per element",
           scale=alt.Scale(domain=[2, 27]), axis=alt.Axis(values=[4, 6, 8, 12, 16, 24]))
YB = alt.Y("gbs:Q", title="achieved bandwidth, GB/s", scale=alt.Scale(domain=[0, 2100]))
TIPS = [alt.Tooltip("operation:N", title="op"),
        alt.Tooltip("implementation:N", title="dtype"),
        alt.Tooltip("b_elem:Q", title="bytes/element"),
        alt.Tooltip("gbs:Q", title="GB/s", format=".0f"),
        alt.Tooltip("gelem_s:Q", title="G elem/s", format=".1f")]

# The 40GB card is drawn hollow: on the ramp the two cards land on top of each
# other, and that coincidence is the finding, so it has to stay visible.
lo, hi = CARDS[0], CARDS[1]
pts_lo = alt.Chart(two.filter(pl.col("card") == lo)).mark_point(
    size=150, filled=False, strokeWidth=2).encode(
    x=XB, y=YB, color=alt.Color("card:N", scale=CSCALE, legend=None), tooltip=TIPS)
pts_hi = alt.Chart(two.filter(pl.col("card") == hi)).mark_point(
    size=80, filled=True, opacity=0.95).encode(
    x=XB, y=YB, color=alt.Color("card:N", scale=CSCALE, legend=None), tooltip=TIPS)
mline = alt.Chart(model).mark_line(strokeWidth=1.4, strokeDash=[5, 3], opacity=0.9).encode(
    x=XB, y=YB, color=alt.Color("card:N", scale=CSCALE, legend=None))
labels = alt.Chart(
    limits.with_columns(pl.lit(17.5).alias("x"), (pl.col("bw") + 75).alias("y"))
).mark_text(align="left", fontSize=11.5, fontWeight=500).encode(
    x="x:Q", y="y:Q", text="card:N", color=alt.Color("card:N", scale=CSCALE, legend=None))
note = alt.Chart(pl.DataFrame({"x": [2.4], "y": [1990.0],
                               "t": ["dashed = min(168 G elem/s × bytes, the card's HBM ceiling)"]})).mark_text(
    align="left", fontSize=11, color=P["ink2"]).encode(x="x:Q", y="y:Q", text="t:N")

fig8 = (mline + pts_lo + pts_hi + labels + note).properties(
    width=500, height=310,
    title=alt.Title("Two ceilings, and bytes per element decides which one binds",
                    subtitle="100M elements · hot, out= · elementwise ops only · the ramp is "
                             "a fixed 168 G elements/s; the plateau is the card's HBM"),
)
figure(fig8, two.sort("card", "b_elem", "gbs"), "Table view — the two ceilings")

alt.LayerChart(...)

operation,implementation,b_elem,us,gbs,gelem_s,card
str,str,i64,f64,f64,f64,str
"""cos""","""GPU fp16""",4,593.104005,674.417976,168.604494,"""40GB (1,555 GB/s)"""
"""exp""","""GPU fp16""",4,592.25601,675.383733,168.845933,"""40GB (1,555 GB/s)"""
"""sqrt""","""GPU fp16""",4,592.17599,675.474874,168.868719,"""40GB (1,555 GB/s)"""
"""div""","""GPU fp16""",6,594.23998,1009.693161,168.282193,"""40GB (1,555 GB/s)"""
"""mul""","""GPU fp16""",6,594.000012,1010.100991,168.350165,"""40GB (1,555 GB/s)"""
"""add""","""GPU fp16""",6,593.344003,1011.217797,168.5363,"""40GB (1,555 GB/s)"""
"""muladd_fused""","""GPU fp16""",6,592.384011,1012.856534,168.809422,"""40GB (1,555 GB/s)"""
"""cos""","""GPU fp32""",8,618.128002,1294.230325,161.778791,"""40GB (1,555 GB/s)"""
"""sqrt""","""GPU fp32""",8,613.471985,1304.053203,163.00665,"""40GB (1,555 GB/s)"""


Every elementwise cell in both runs falls on one of two limits.

On the ramp, the kernel retires **168 G elements per second** — 168.0 to 168.6 on
the 80GB card, 167.0 to 168.8 on the 40GB card — no matter which operation it is
or which dtype. `sqrt`, `exp`, `cos`, `add`, `mul`, `div` and `muladd_fused` all
sit on it. Achieved bandwidth along the ramp is simply that rate times the bytes
each element carries, which is why fp16 `add` (6 bytes) tops out at
168 × 6 ≈ 1 008 GB/s, or 49.5% of peak. fp16 is not failing to saturate the bus;
it is running out of elements per second while carrying too few bytes each.

On the plateau, the kernel is limited by HBM: 1 767 GB/s on the 80GB card (87% of
peak), 1 380 GB/s on the 40GB (89%).

The two cards make this a controlled experiment, since they have identical compute
and differ only in memory. The element ceiling is unchanged between them —
167 against 168 G elem/s — while the bandwidth ceiling rises by 28%, and the
crossover consequently moves from 8.3 to 10.5 bytes per element. That is why
`float32 exp` (8 bytes/element) is bandwidth-bound on the 40GB card and
element-bound on the 80GB one, and it is the reason the two cards disagree about
what precision costs.

In [13]:
ratios = (
    D["gpu_sweep80"].filter(OUT_HOT & (pl.col("n") == 25_000_000))
    .pivot("implementation", index="operation", values="us").drop_nulls()
    .with_columns((pl.col("GPU fp64") / pl.col("GPU fp32")).alias("fp64 / fp32"),
                  (pl.col("GPU fp32") / pl.col("GPU fp16")).alias("fp32 / fp16"))
    .sort("fp32 / fp16", descending=True)
)
rlong = ratios.unpivot(["fp64 / fp32", "fp32 / fp16"], index="operation",
                       variable_name="pair", value_name="ratio")
PAIR = ["fp64 / fp32", "fp32 / fp16"]

rbars = alt.Chart(rlong).mark_bar(size=9).encode(
    y=alt.Y("operation:N", title=None,
            sort=alt.EncodingSortField("ratio", op="max", order="descending")),
    yOffset=alt.YOffset("pair:N", sort=PAIR),
    x=alt.X("ratio:Q", title="time ratio", scale=alt.Scale(domain=[0, 2.3], nice=False)),
    color=alt.Color("pair:N", sort=PAIR,
                    scale=alt.Scale(domain=PAIR, range=P["series"][:2]),
                    legend=alt.Legend(title=None, orient="top")),
    tooltip=[alt.Tooltip("operation:N", title="op"), alt.Tooltip("pair:N", title="ratio"),
             alt.Tooltip("ratio:Q", format=".2f")])
rval = rbars.mark_text(align="left", dx=4, fontSize=10, color=P["ink2"]).encode(
    text=alt.Text("ratio:Q", format=".2f"))
two_rule = alt.Chart(pl.DataFrame({"x": [2.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(x="x:Q")
two_txt = alt.Chart(pl.DataFrame({"x": [2.0], "t": ["2.0 — half the bytes, half the time"]})).mark_text(
    align="right", dx=-5, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(300), text="t:N")

fig9 = (rbars + rval + two_rule + two_txt).properties(
    width=420, height=320,
    title=alt.Title("Halving precision pays only where the op is bandwidth-bound",
                    subtitle="gpu_sweep80 · 100M elements · hot, out= · only the fused matmul "
                             "clears 2.0; the unary ops get nothing from fp16"),
)
figure(fig9, ratios, "Table view — precision ratios")

alt.LayerChart(...)

operation,GPU fp16,GPU fp32,GPU fp64,fp64 / fp32,fp32 / fp16
str,f64,f64,f64,f64,f64
"""matmul_explicit_fused""",374.607995,685.855985,1454.991996,2.121425,1.830863
"""matmul_explicit""",3398.191929,5436.70392,10368.031979,1.907044,1.599881
"""muladd""",1177.343965,1477.199972,2704.783916,1.831021,1.254689
"""add""",594.815999,743.423998,1358.448029,1.827286,1.249839
"""muladd_fused""",593.103975,740.175992,1359.27999,1.836428,1.24797
"""div""",595.423996,740.960002,1401.311994,1.891211,1.244424
"""mul""",595.216006,740.208,1360.607982,1.838143,1.243596
"""sqrt""",594.352007,595.407993,934.527993,1.569559,1.001777
"""cos""",594.864011,594.720006,948.112011,1.594216,0.999758


The ratios follow from the previous figure rather than from the byte counts.

`matmul_explicit_fused` is the only operation to beat 2.0 in either column (2.12
and 1.83): it is a single fused kernel doing real arithmetic per element, so it is
the one case where the A100's native fp16 units are the resource in play.

The elementwise binary ops give 1.83–1.84 for fp64/fp32 and 1.24–1.25 for
fp32/fp16. Neither is the naive 2.0, and both are predicted by the ceilings:
2 × (80.1/86.6) = 1.85 and 2 × (49.6/80.1) = 1.24.

The unary ops give exactly 1.00 for fp32/fp16. `sqrt`, `exp` and `cos` in fp32
move 8 bytes/element and in fp16 move 4, and both are below the crossover, so both
run at the same 168 G elem/s and take the same 594 µs. Halving the precision of a
memory-light elementwise kernel on this card buys nothing at all.

An earlier version of this notebook reported fp64/fp32 as "2.00 for every
elementwise operation." That was measured on the 40GB card, where the lower
bandwidth ceiling put every op with 8 or more bytes per element on the plateau, so
the ratio really was the byte ratio. The claim was true of that card and is not a
general fact about fp64.

### The two machines on one axis

`add` on a 64-bit float, streaming, into a preallocated buffer, is the one
configuration both machines run. Both are now resolved finely enough to say where
the crossover is rather than that it was not found.

In [14]:
cross = pl.concat([
    D["cpu_dense"].filter(OUT_COLD & (pl.col("operation") == "add")
                          & (pl.col("implementation") == "float64"))
    .with_columns(pl.lit("x86 CPU, 1 thread (float64)").alias("machine")),
    D["gpu_dense80"].filter(OUT_COLD & (pl.col("operation") == "add")
                            & (pl.col("implementation") == "GPU fp64"))
    .with_columns(pl.lit("A100-80GB (fp64)").alias("machine")),
]).select("machine", "n_elem", "ns_elem", "us", "gbs").sort("machine", "n_elem")
MACH = ["x86 CPU, 1 thread (float64)", "A100-80GB (fp64)"]
XC = alt.X("n_elem:Q", scale=alt.Scale(type="log", nice=False), title="elements (log)",
           axis=alt.Axis(format="~s", values=[1000, 20_000, 400_000, 8_000_000, 250_000_000]))
YC = alt.Y("ns_elem:Q", scale=alt.Scale(type="log"), title="ns per element (log)",
           axis=alt.Axis(values=[0.01, 0.1, 1, 10], format="~g"))
CC = alt.Color("machine:N", sort=MACH, scale=alt.Scale(domain=MACH, range=P["series"][:2]),
               legend=alt.Legend(title=None, orient="top"))

xline = alt.Chart(pl.DataFrame({"x": [19_800.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1, strokeDash=[4, 3]).encode(x="x:Q")
xtext = alt.Chart(pl.DataFrame({"x": [19_800.0], "t": ["crossover ≈ 19 800 elements"]})).mark_text(
    align="left", dx=7, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(14), text="t:N")
cl = alt.Chart(cross).mark_line(point=alt.OverlayMarkDef(size=45)).encode(
    x=XC, y=YC, color=CC,
    tooltip=[alt.Tooltip("machine:N", title="machine"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".4f"),
             alt.Tooltip("us:Q", title="us/call", format=".2f")])

fig10 = (cl + xline + xtext).properties(
    width=500, height=310,
    title=alt.Title("The same operation on both machines",
                    subtitle="add · 64-bit float · cold, out= · below 20 000 elements the "
                             "GPU's per-call floor makes the CPU faster"),
)
figure(fig10, cross, "Table view — CPU against GPU")

alt.LayerChart(...)

machine,n_elem,ns_elem,us,gbs
str,i64,f64,f64,f64
"""A100-80GB (fp64)""",1000,8.989714,8.989714,2.669894
"""A100-80GB (fp64)""",2000,4.2115,8.423,5.698682
"""A100-80GB (fp64)""",4000,2.188,8.752,10.970984
"""A100-80GB (fp64)""",10000,0.8525,8.525,28.152698
"""A100-80GB (fp64)""",20000,0.440457,8.809143,54.505811
"""A100-80GB (fp64)""",40000,0.221514,8.860572,108.362824
"""A100-80GB (fp64)""",100000,0.087783,8.778286,273.58925
"""A100-80GB (fp64)""",200000,0.044203,8.840615,542.956302
"""A100-80GB (fp64)""",400000,0.022622,9.048727,1060.924046


The CPU is faster per element below roughly 19 800 elements — 160 KB of float64 —
because the GPU is still paying its flat 8.4 µs per call while the CPU is near the
bottom of its cache curve at 0.37 ns/element. Above that the GPU wins, by 57× at
the largest sizes, where its advantage stops growing because both machines are
then bandwidth-bound on their own memory.

---

## Conclusions

1. **Three points cannot describe a cache.** Resolving the size axis turned the
   CPU's single U into two knees and a DRAM plateau, and reversed the reading of
   the `hot`/`cold` gap: it peaks at 1.49× inside the L3 band and closes to 1.00×
   once both variants stream from memory.

2. **Precision is not an evenly spaced dial.** Ten hardware dtypes sit within a
   small factor of `float64`; x87 `longdouble` is 3–12× away and software
   `quad-sleef` 27–121×. There is no intermediate regime between arithmetic the
   FPU performs and arithmetic a library performs.

3. **`float16` on this CPU is a conversion cost wearing an arithmetic label.** It
   costs 6.0 ns/element at every size from 400 to 8 million elements, has no cache
   behaviour, and is unimproved by doing the widening by hand — which places the
   expense in the `float16`↔`float32` conversion, at roughly 15× the cost of a
   plain copy, on a CPU whose `F16C` instructions exist to make it cheap.

4. **The allocator has a cliff, not a slope.** A result buffer at or below
   30.52 MiB has a median allocation cost of zero and is often negative; one at
   61.04 MiB costs 11–54%, median 34%. The boundary is glibc's 32 MiB `mmap`
   threshold cap.
   An earlier reading attributed this to result *width*, which at a single problem
   size is indistinguishable from which side of the threshold the buffer fell on.

5. **A cold destination is a real cost, but not the one it was blamed for.**
   Cycling the destination through a pool costs 40–60% while the result still fits
   in cache and nothing once it does not, which rules it out as the explanation
   for the allocating form being faster at intermediate sizes. That residue
   remains unexplained.

6. **The GPU has two ceilings and the byte width picks one.** Elementwise kernels
   retire 168 G elements/s or move 1 767 GB/s, whichever binds first, crossing at
   10.5 bytes per element. Everything else in Part 3 follows: fp16 plateaus at
   half the bandwidth utilisation of fp64, halving precision buys 1.25× on binary
   ops and exactly nothing on unary ones, and only the fused matmul — the one
   genuinely compute-bound kernel — sees the full 2×.

7. **The two A100s are the control.** Identical compute, 31% more bandwidth: the
   element ceiling stays at 167–168 G elem/s and the crossover moves from 8.3 to
   10.5 bytes/element. Precision ratios measured on one card do not transfer to
   the other, which is why the earlier "fp64 costs exactly 2× fp32" was a fact
   about a 40GB A100 rather than about fp64.

### Limitations

- The unexplained residue in Part 2 — the allocating form beating `out=` by up to
  32% between 1 and 17 MiB — is not settled by anything here. Forcing
  `MALLOC_MMAP_THRESHOLD_` to either extreme would say whether the allocator path
  is involved at all.
- More repeats did not reduce spread: `cpu_dram` at 20 repeats and `cpu_dram50` at
  50 have median IQRs of 3.05% and 3.15%. The noise is structural, so the
  interquartile range carried through `cells()` as `us_lo`/`us_hi` is the honest
  error bar, and it is wide on a minority of cells.
- `cpu_dense` skipped 195 of 2 160 cells and `cpu_dram50` 52 of 200, in both cases
  the slowest ones, under `--max-call-ms 50`. Aggregates over a dtype or an
  operation are biased fast unless restricted to complete cells.
- CPU timings are single-threaded on one NUMA node, and use `perf_counter` against
  the GPU's CUDA events. The cross-machine figure compares end-to-end call cost,
  not kernel time.